# TelecomTS Model Evaluation

Evaluate **Mantis** (random-init and pretrained), and **TimesNet** on the [TelecomTS](https://huggingface.co/datasets/AliMaatouk/TelecomTS) dataset for two tasks:

1. **Anomaly Detection** — binary classification (anomaly vs. normal)
2. **Root Cause Analysis** — multi-class classification (10 synthetic anomaly types)

Uses the [TelecomTS benchmark pipeline](https://github.com/Ali-maatouk/TelecomTS)
with the exact encoder architectures and hyperparameters from the paper.

Reference: [APPENG-5739](https://redhat.atlassian.net/browse/APPENG-5739)

## 1. Setup

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps", "git+https://github.com/vfeofanov/mantis.git",
])

In [ ]:
!git clone --depth 1 https://github.com/Ali-maatouk/TelecomTS.git /tmp/TelecomTS 2>/dev/null

import sys
sys.path.insert(0, "/tmp/TelecomTS/src")

In [ ]:
import yaml
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    ConfusionMatrixDisplay,
)

from utils.data_utils import preprocess
from utils.train_utils import prepare, evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Load TelecomTS Dataset

32K time series samples from a 5G testbed.  
18 KPI channels (PHY, MAC, Network layers), 128 timesteps each, sampled at 10 Hz.

In [ ]:
dataset = load_dataset(
    "AliMaatouk/TelecomTS",
    data_files={"full": "**/chunked.jsonl"},
)["full"]

splits = dataset.train_test_split(test_size=0.2, seed=SEED)
train_data = list(splits["train"])
test_data = list(splits["test"])

random.shuffle(train_data)

print(f"Train: {len(train_data):,} samples")
print(f"Test:  {len(test_data):,} samples")
print(f"Total: {len(train_data) + len(test_data):,} samples")

In [ ]:
sample = train_data[0]
print(f"Keys: {list(sample.keys())}")
print(f"KPIs: {list(sample['KPIs'].keys())}")
print(f"Anomaly exists: {sample['anomalies']['exists']}")
print(f"Anomaly type: {sample['anomalies'].get('type', 'N/A')}")
print(f"Labels: {sample['labels']}")

## 3. Config

Hyperparameters from the [TelecomTS config](https://github.com/Ali-maatouk/TelecomTS/blob/main/configs/config.yaml).

In [ ]:
with open("/tmp/TelecomTS/configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

TRAIN_CFG = config["train"]
TIMESNET_CFG = config["TimesNet_model"]
MANTIS_CFG = config["Mantis_model"]

print("Training config:", TRAIN_CFG)
print("\nTimesNet config:", TIMESNET_CFG)
print("\nMantis config:", MANTIS_CFG)

## 4. Training + Evaluation Helpers

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


def make_config(encoder_type, task_type, base_config):
    """Build a config dict matching TelecomTS's prepare() expectations."""
    return {
        "encoder_type": encoder_type,
        "task_type": task_type,
        "seed": SEED,
        "train": base_config["train"],
        f"{encoder_type}_model": base_config[f"{encoder_type}_model"],
    }


def compute_class_weights(y):
    """Compute inverse-frequency class weights for imbalanced datasets."""
    counts = np.bincount(y)
    weights = 1.0 / counts.astype(np.float64)
    weights = weights / weights.sum() * len(counts)
    return torch.tensor(weights, dtype=torch.float32)


class FocalLoss(nn.Module):
    """Focal loss — downweights easy examples to focus on hard ones."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def train_model(cfg, X_train, y_train, epochs=None, balance=None, focal_gamma=2.0, lr=None):
    """Train an encoder + head using TelecomTS's prepare() utility.

    balance: None (no rebalancing), "weights" (class-weighted CE),
             or "focal" (focal loss with class weights).
    focal_gamma: gamma for focal loss (lower = gentler, default 2.0).
    lr: override learning rate (default uses config value).
    """
    model, head, train_dataset, train_dataloader, optimizer, criterion = prepare(
        cfg, X_train, y_train
    )

    if lr is not None:
        optimizer = torch.optim.Adam(
            list(model.parameters()) + list(head.parameters()),
            lr=lr,
            weight_decay=cfg["train"]["optim"]["weight_decay"],
            betas=cfg["train"]["optim"]["betas"],
        )

    if balance is not None:
        w = compute_class_weights(y_train).to(DEVICE)
        if balance == "weights":
            criterion = nn.CrossEntropyLoss(weight=w)
        elif balance == "focal":
            criterion = FocalLoss(alpha=w, gamma=focal_gamma)

    model = model.to(DEVICE)
    head = head.to(DEVICE)

    if epochs is None:
        epochs = cfg["train"]["epochs"]

    model.train()
    head.train()

    for epoch in range(epochs):
        losses = []
        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
            optimizer.zero_grad()
            outputs = model(x.permute(0, 2, 1))
            logits = head(outputs)
            loss = criterion(logits, y)
            losses.append(loss.item())
            loss.backward()
            optimizer.step()

        print(f"  Epoch {epoch+1}/{epochs} — loss: {np.mean(losses):.4f}")

    return model, head, train_dataset


def predict(model, head, X, batch_size=64):
    """Get predictions from encoder + head."""
    model.eval()
    head.eval()
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.zeros(len(X), dtype=torch.long),
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            out = head(model(xb.permute(0, 2, 1)))
            preds.append(out.argmax(dim=1).cpu().numpy())

    return np.concatenate(preds)

In [ ]:
from mantis.architecture import MantisV1


class PretrainedMantisEncoder(nn.Module):
    """Mantis-V1 with pretrained backbone (paris-noah/Mantis-8M).

    Same per-channel processing and mean-pooling as the TelecomTS wrapper,
    but starts from pretrained weights (8.1M params, hidden_dim=256)
    instead of random initialization (hidden_dim=64).
    """

    PRETRAINED_SEQ_LEN = 512
    PRETRAINED_HIDDEN_DIM = 256

    def __init__(self, checkpoint="paris-noah/Mantis-8M", d_model=64, dropout=0.1):
        super().__init__()
        self.backbone = MantisV1(device="cpu")
        self.backbone = self.backbone.from_pretrained(checkpoint)
        self.act = F.gelu
        self.dropout = nn.Dropout(dropout)
        self.projection = nn.Linear(self.PRETRAINED_HIDDEN_DIM, d_model)

    def forward(self, x_enc):
        """x_enc: [B, T, C] — same convention as TelecomTS Mantis."""
        x = x_enc.transpose(1, 2).contiguous()  # [B, C, T]
        B, C, T = x.shape
        x = x.reshape(B * C, 1, T)
        if T != self.PRETRAINED_SEQ_LEN:
            x = F.interpolate(x, size=self.PRETRAINED_SEQ_LEN, mode="linear", align_corners=False)
        h = self.backbone(x)           # [B*C, 256]
        h = h.reshape(B, C, -1)        # [B, C, 256]
        h = h.mean(dim=1)              # [B, 256]
        h = self.dropout(self.act(h))
        return self.projection(h)      # [B, d_model]


def train_pretrained_mantis(X_train, y_train, task_type,
                            checkpoint="paris-noah/Mantis-8M",
                            d_model=64, epochs=15, lr=1e-4, batch_size=64,
                            balance=None, focal_gamma=2.0):
    """Train pretrained Mantis encoder + classification head.

    Uses a lower default LR (1e-4) than random-init (1e-3) since the
    backbone already has learned representations.
    """
    encoder = PretrainedMantisEncoder(
        checkpoint=checkpoint, d_model=d_model, dropout=0.1
    ).to(DEVICE)

    n_classes = 2 if task_type == "anomaly detection" else 10
    if task_type == "anomaly detection":
        head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, n_classes))
    else:
        head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(0.2), nn.Linear(d_model, n_classes))
    head = head.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    if balance is not None:
        w = compute_class_weights(y_train).to(DEVICE)
        if balance == "weights":
            criterion = nn.CrossEntropyLoss(weight=w)
        elif balance == "focal":
            criterion = FocalLoss(alpha=w, gamma=focal_gamma)

    optimizer = torch.optim.Adam(
        list(encoder.parameters()) + list(head.parameters()), lr=lr
    )

    ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

    encoder.train()
    head.train()
    for epoch in range(epochs):
        losses = []
        for x, y in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = head(encoder(x.permute(0, 2, 1)))
            loss = criterion(out, y)
            losses.append(loss.item())
            loss.backward()
            optimizer.step()
        print(f"  Epoch {epoch+1}/{epochs} — loss: {np.mean(losses):.4f}")

    return encoder, head

---
## 5. Task 1 — Anomaly Detection (Binary)

Classify each 128-timestep sample as **normal** (0) or **anomalous** (1).

The dataset is heavily imbalanced (~96% normal, ~4% anomaly), so we use
**inverse-frequency class weights** in the loss function to penalize
misclassifying the minority class more heavily. Switch to `balance="focal"`
for focal loss, or `balance=None` for unweighted baseline.

In [ ]:
X_train_ad, y_train_ad = preprocess(train_data, "anomaly detection")
X_test_ad, y_test_ad = preprocess(test_data, "anomaly detection")

print(f"Train: {X_train_ad.shape}, labels: {np.bincount(y_train_ad)}")
print(f"Test:  {X_test_ad.shape}, labels: {np.bincount(y_test_ad)}")

### 5a. Mantis — Anomaly Detection

In [ ]:
cfg_mantis_ad = make_config("Mantis", "anomaly detection", config)

print("Training Mantis for anomaly detection...")
mantis_ad_model, mantis_ad_head, _ = train_model(cfg_mantis_ad, X_train_ad, y_train_ad, epochs=15, balance="weights")

y_pred_mantis_ad = predict(mantis_ad_model, mantis_ad_head, X_test_ad)

print("\n=== Mantis — Anomaly Detection ===")
print(classification_report(y_test_ad, y_pred_mantis_ad, target_names=["Normal", "Anomaly"]))

### 5b. TimesNet — Anomaly Detection

In [ ]:
cfg_timesnet_ad = make_config("TimesNet", "anomaly detection", config)

print("Training TimesNet for anomaly detection...")
timesnet_ad_model, timesnet_ad_head, _ = train_model(cfg_timesnet_ad, X_train_ad, y_train_ad, epochs=15, balance="weights")

y_pred_timesnet_ad = predict(timesnet_ad_model, timesnet_ad_head, X_test_ad)

print("\n=== TimesNet — Anomaly Detection ===")
print(classification_report(y_test_ad, y_pred_timesnet_ad, target_names=["Normal", "Anomaly"]))

### 5c. Mantis (Pretrained) — Anomaly Detection

Uses the `paris-noah/Mantis-8M` pretrained backbone (8.1M params, hidden_dim=256)
instead of random initialization (hidden_dim=64). The backbone was pretrained on
CauKer 2M, a large-scale synthetic time series dataset. Lower LR (1e-4) for
fine-tuning vs 1e-3 for training from scratch.

In [ ]:
print("Training Mantis (pretrained) for anomaly detection...")
mantis_pt_ad_encoder, mantis_pt_ad_head = train_pretrained_mantis(
    X_train_ad, y_train_ad, "anomaly detection",
    epochs=15, lr=1e-4, balance="weights",
)

y_pred_mantis_pt_ad = predict(mantis_pt_ad_encoder, mantis_pt_ad_head, X_test_ad)

print("\n=== Mantis (Pretrained) — Anomaly Detection ===")
print(classification_report(y_test_ad, y_pred_mantis_pt_ad, target_names=["Normal", "Anomaly"]))

---
## 6. Task 2 — Root Cause Analysis (Multi-class)

For anomalous samples only: classify the cause among 10 synthetic anomaly types.  
(Jamming is excluded — it's the single real anomaly and treated separately.)

In [ ]:
RCA_CLASSES = [
    "Antenna Failure",
    "Co-Channel Interference (Mild)",
    "Co-Channel Interference (Severe)",
    "Faulty RF Filters (Temporal)",
    "Doppler Shift (Severe)",
    "Faulty Handover Algorithm (Too Frequent)",
    "Buffer Overflow (Gradual Buildup)",
    "Resource Allocation Bugs",
    "High Network Congestion (Gradual Buildup)",
    "High Network Congestion (Sudden Spike)",
]

X_train_rca, y_train_rca = preprocess(train_data, "root-cause analysis")
X_test_rca, y_test_rca = preprocess(test_data, "root-cause analysis")

print(f"Train: {X_train_rca.shape}, class distribution: {np.bincount(y_train_rca)}")
print(f"Test:  {X_test_rca.shape}, class distribution: {np.bincount(y_test_rca)}")

### 6a. Mantis — Root Cause Analysis

In [ ]:
cfg_mantis_rca = make_config("Mantis", "root-cause analysis", config)

print("Training Mantis for root cause analysis...")
mantis_rca_model, mantis_rca_head, _ = train_model(cfg_mantis_rca, X_train_rca, y_train_rca, epochs=15)

y_pred_mantis_rca = predict(mantis_rca_model, mantis_rca_head, X_test_rca)

print("\n=== Mantis — Root Cause Analysis ===")
print(classification_report(y_test_rca, y_pred_mantis_rca, target_names=RCA_CLASSES))

### 6b. TimesNet — Root Cause Analysis

In [ ]:
cfg_timesnet_rca = make_config("TimesNet", "root-cause analysis", config)

print("Training TimesNet for root cause analysis...")
timesnet_rca_model, timesnet_rca_head, _ = train_model(cfg_timesnet_rca, X_train_rca, y_train_rca, epochs=35)

y_pred_timesnet_rca = predict(timesnet_rca_model, timesnet_rca_head, X_test_rca)

print("\n=== TimesNet — Root Cause Analysis ===")
print(classification_report(y_test_rca, y_pred_timesnet_rca, target_names=RCA_CLASSES))

### 6c. Mantis (Pretrained) — Root Cause Analysis

In [ ]:
print("Training Mantis (pretrained) for root cause analysis...")
mantis_pt_rca_encoder, mantis_pt_rca_head = train_pretrained_mantis(
    X_train_rca, y_train_rca, "root-cause analysis",
    epochs=35, lr=1e-4,
)

y_pred_mantis_pt_rca = predict(mantis_pt_rca_encoder, mantis_pt_rca_head, X_test_rca)

print("\n=== Mantis (Pretrained) — Root Cause Analysis ===")
print(classification_report(y_test_rca, y_pred_mantis_pt_rca, target_names=RCA_CLASSES))

---
## 7. Results Comparison

In [ ]:
results = {}
for task, y_true, preds in [
    ("Anomaly Detection", y_test_ad, [
        ("Mantis", y_pred_mantis_ad),
        ("Mantis (Pretrained)", y_pred_mantis_pt_ad),
        ("TimesNet", y_pred_timesnet_ad),
    ]),
    ("Root Cause Analysis", y_test_rca, [
        ("Mantis", y_pred_mantis_rca),
        ("Mantis (Pretrained)", y_pred_mantis_pt_rca),
        ("TimesNet", y_pred_timesnet_rca),
    ]),
]:
    results[task] = {}
    for model_name, y_pred in preds:
        results[task][model_name] = {
            "Accuracy": accuracy_score(y_true, y_pred),
            "F1 (macro)": f1_score(y_true, y_pred, average='macro', zero_division=0),
            "F1 (weighted)": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        }

for task in results:
    print(f'\n=== {task} ===')
    for model_name, metrics in results[task].items():
        print(f'  {model_name}: ' + ', '.join(f'{k}={v:.3f}' for k, v in metrics.items()))

In [ ]:
MODELS = ["Mantis", "Mantis (Pretrained)", "TimesNet"]
COLORS = {"Mantis": "#2196F3", "Mantis (Pretrained)": "#4CAF50", "TimesNet": "#FF9800"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, task in zip(axes, ["Anomaly Detection", "Root Cause Analysis"]):
    metrics = list(results[task]["Mantis"].keys())
    x = np.arange(len(metrics))
    width = 0.25
    for i, model in enumerate(MODELS):
        vals = [results[task][model][m] for m in metrics]
        ax.bar(x + (i - 1) * width, vals, width, label=model, color=COLORS[model])
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.set_ylabel("Score")
    ax.set_title(task)
    ax.legend()
    ax.set_ylim(0, 1)

plt.suptitle("Model Comparison — TelecomTS", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(24, 16))

cm_data = [
    (y_test_ad, y_pred_mantis_ad, "Mantis — AD"),
    (y_test_ad, y_pred_mantis_pt_ad, "Mantis (Pretrained) — AD"),
    (y_test_ad, y_pred_timesnet_ad, "TimesNet — AD"),
    (y_test_rca, y_pred_mantis_rca, "Mantis — RCA"),
    (y_test_rca, y_pred_mantis_pt_rca, "Mantis (Pretrained) — RCA"),
    (y_test_rca, y_pred_timesnet_rca, "TimesNet — RCA"),
]

for ax, (y_true, y_pred, title) in zip(axes.flat, cm_data):
    if "RCA" in title:
        labels = [c[:20] for c in RCA_CLASSES]
    else:
        labels = ["Normal", "Anomaly"]

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title, fontsize=12)
    if "RCA" in title:
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_yticklabels(labels, fontsize=8)

plt.suptitle("Confusion Matrices", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. CPU Inference Benchmark

Measure single-sample and batch inference latency on CPU to verify the
pretrained Mantis model is fast enough for production deployment without GPU.

In [ ]:
import time

cpu_encoder = mantis_pt_ad_encoder.cpu().eval()
cpu_head = mantis_pt_ad_head.cpu().eval()

WARMUP = 10
RUNS = 100

# --- Single-sample latency ---
x_single = torch.randn(1, 18, 128)

for _ in range(WARMUP):
    with torch.no_grad():
        cpu_head(cpu_encoder(x_single.permute(0, 2, 1)))

times = []
for _ in range(RUNS):
    start = time.perf_counter()
    with torch.no_grad():
        cpu_head(cpu_encoder(x_single.permute(0, 2, 1)))
    times.append(time.perf_counter() - start)

single_ms = np.array(times) * 1000
print(f"Single sample (1):  mean={single_ms.mean():.1f} ms, "
      f"median={np.median(single_ms):.1f} ms, "
      f"p95={np.percentile(single_ms, 95):.1f} ms")

# --- Batch latency ---
for batch_size in [8, 32, 64]:
    x_batch = torch.randn(batch_size, 18, 128)

    for _ in range(WARMUP):
        with torch.no_grad():
            cpu_head(cpu_encoder(x_batch.permute(0, 2, 1)))

    times = []
    for _ in range(RUNS):
        start = time.perf_counter()
        with torch.no_grad():
            cpu_head(cpu_encoder(x_batch.permute(0, 2, 1)))
        times.append(time.perf_counter() - start)

    batch_ms = np.array(times) * 1000
    per_sample = batch_ms / batch_size
    print(f"Batch ({batch_size:2d}):         mean={batch_ms.mean():.1f} ms total, "
          f"{per_sample.mean():.1f} ms/sample, "
          f"p95={np.percentile(batch_ms, 95):.1f} ms")

---
## 9. Export Model Weights

Save the fine-tuned pretrained Mantis encoder and classification heads for
both tasks. These can be loaded directly for inference without retraining.

In [ ]:
import os

EXPORT_DIR = "../models"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Anomaly Detection
torch.save({
    "encoder": mantis_pt_ad_encoder.state_dict(),
    "head": mantis_pt_ad_head.state_dict(),
    "task": "anomaly detection",
    "n_classes": 2,
    "checkpoint": "paris-noah/Mantis-8M",
    "d_model": 64,
    "epochs": 15,
    "lr": 1e-4,
    "balance": "weights",
}, os.path.join(EXPORT_DIR, "mantis_pretrained_ad.pt"))

# Root Cause Analysis
torch.save({
    "encoder": mantis_pt_rca_encoder.state_dict(),
    "head": mantis_pt_rca_head.state_dict(),
    "task": "root-cause analysis",
    "n_classes": 10,
    "checkpoint": "paris-noah/Mantis-8M",
    "d_model": 64,
    "epochs": 35,
    "lr": 1e-4,
    "balance": None,
}, os.path.join(EXPORT_DIR, "mantis_pretrained_rca.pt"))

for f in os.listdir(EXPORT_DIR):
    size_mb = os.path.getsize(os.path.join(EXPORT_DIR, f)) / (1024 * 1024)
    print(f"{f}: {size_mb:.1f} MB")

---
## 10. Log to MLFlow

Register the fine-tuned models with MLFlow on RHOAI for serving and testing.
Artifacts are stored in MinIO (`minio-mlflow` service in the namespace).

In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "mlflow", "boto3",
])

In [ ]:
import mlflow
import mlflow.pyfunc

MLFLOW_URL = "https://rh-ai.apps.ai-dev02.kni.syseng.devcluster.openshift.com/mlflow"

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URL
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio-mlflow.telcomts-evaluation.svc:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "mlflow"
os.environ["AWS_SECRET_ACCESS_KEY"] = "mlflow123"

sa_token_path = "/var/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(sa_token_path):
    with open(sa_token_path) as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()

class MantisPretrainedModel(mlflow.pyfunc.PythonModel):
    """MLFlow wrapper for fine-tuned pretrained Mantis. Reconstructs the
    encoder + head from a saved checkpoint and runs inference on CPU."""

    def load_context(self, context):
        import torch
        import torch.nn as nn
        import torch.nn.functional as F
        from mantis.architecture import MantisV1

        self.torch = torch
        ckpt = torch.load(context.artifacts["weights"], map_location="cpu", weights_only=False)
        self.n_classes = ckpt["n_classes"]
        d_model = ckpt["d_model"]

        class Encoder(nn.Module):
            def __init__(self):
                super().__init__()
                self.backbone = MantisV1(device="cpu")
                self.backbone = self.backbone.from_pretrained(ckpt["checkpoint"])
                self.act = F.gelu
                self.dropout = nn.Dropout(0.1)
                self.projection = nn.Linear(256, d_model)

            def forward(self, x_enc):
                x = x_enc.transpose(1, 2).contiguous()
                B, C, T = x.shape
                x = x.reshape(B * C, 1, T)
                if T != 512:
                    x = F.interpolate(x, size=512, mode="linear", align_corners=False)
                h = self.backbone(x).reshape(B, C, -1).mean(dim=1)
                return self.projection(self.dropout(self.act(h)))

        self.encoder = Encoder()
        self.encoder.load_state_dict(ckpt["encoder"])
        self.encoder.eval()

        if ckpt["task"] == "anomaly detection":
            self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, self.n_classes))
        else:
            self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(0.2), nn.Linear(d_model, self.n_classes))
        self.head.load_state_dict(ckpt["head"])
        self.head.eval()

    def predict(self, context, model_input):
        import numpy as np

        x_np = model_input.values if hasattr(model_input, "values") else np.array(model_input)
        if x_np.ndim == 2 and x_np.shape[1] == 18 * 128:
            x_np = x_np.reshape(-1, 18, 128)

        x = self.torch.tensor(x_np, dtype=self.torch.float32)
        with self.torch.no_grad():
            return self.head(self.encoder(x.permute(0, 2, 1))).argmax(dim=1).numpy()

In [ ]:
mlflow.set_experiment("telecomts-anomaly-detection")

conda_env = {
    "channels": ["defaults"],
    "dependencies": [
        f"python={sys.version_info.major}.{sys.version_info.minor}",
        "pip",
        {"pip": [
            "torch",
            "numpy",
            "mantis-tsfm",
        ]},
    ],
}

with mlflow.start_run(run_name="mantis-pretrained-telecomts") as run:
    # Log training parameters
    mlflow.log_params({
        "checkpoint": "paris-noah/Mantis-8M",
        "d_model": 64,
        "ad_epochs": 15,
        "ad_lr": 1e-4,
        "ad_balance": "weights",
        "rca_epochs": 35,
        "rca_lr": 1e-4,
        "dataset": "AliMaatouk/TelecomTS",
        "train_test_split": 0.2,
        "seed": 42,
    })

    # Log metrics
    for task_key, task_name in [("Anomaly Detection", "ad"), ("Root Cause Analysis", "rca")]:
        for metric, value in results[task_key]["Mantis (Pretrained)"].items():
            mlflow.log_metric(f"{task_name}_{metric.lower().replace(' ', '_').replace('(', '').replace(')', '')}", value)

    # Log AD model
    mlflow.pyfunc.log_model(
        artifact_path="mantis_ad",
        python_model=MantisPretrainedModel(),
        artifacts={"weights": os.path.join(EXPORT_DIR, "mantis_pretrained_ad.pt")},
        conda_env=conda_env,
        registered_model_name="mantis-pretrained-ad",
    )

    # Log RCA model
    mlflow.pyfunc.log_model(
        artifact_path="mantis_rca",
        python_model=MantisPretrainedModel(),
        artifacts={"weights": os.path.join(EXPORT_DIR, "mantis_pretrained_rca.pt")},
        conda_env=conda_env,
        registered_model_name="mantis-pretrained-rca",
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    print("Models registered: mantis-pretrained-ad, mantis-pretrained-rca")

### Test Models from MLFlow Registry

In [ ]:
# Load models from MLFlow registry
ad_model = mlflow.pyfunc.load_model("models:/mantis-pretrained-ad/latest")
rca_model = mlflow.pyfunc.load_model("models:/mantis-pretrained-rca/latest")

# Test AD on 10 samples
ad_preds = ad_model.predict(X_test_ad[:10])
print("=== Anomaly Detection (10 samples) ===")
for i, (pred, actual) in enumerate(zip(ad_preds, y_test_ad[:10])):
    label = "Anomaly" if pred == 1 else "Normal"
    actual_label = "Anomaly" if actual == 1 else "Normal"
    match = "✓" if pred == actual else "✗"
    print(f"  Sample {i}: predicted={label}, actual={actual_label} {match}")

# Test RCA on 10 samples
rca_preds = rca_model.predict(X_test_rca[:10])
print("\n=== Root Cause Analysis (10 samples) ===")
for i, (pred, actual) in enumerate(zip(rca_preds, y_test_rca[:10])):
    pred_name = RCA_CLASSES[pred]
    actual_name = RCA_CLASSES[actual]
    match = "✓" if pred == actual else "✗"
    print(f"  Sample {i}: predicted={pred_name}, actual={actual_name} {match}")